# Coursework 1

In [1]:
using JLD2
using LinearAlgebra

## 1. Your Data 

### Data Type Declaration

The data in the provided data file are stored using the data types defined below. These type definitions are required to load the data correctly.

**Do not modify these definitions.**

In [2]:
# Data type for Q7_1
#########################
# Handling Large Integers
#########################
mutable struct QLargeInt
   a::Int64 
   b::Int64 
   n::Int64 
   c1 :: Union{Missing,Int}
   c2 :: Union{Missing,Int}
   c3 :: Union{Missing,Int}
end 

# Data type for Q7_2
#########################
# Congruence Inverse: Integer
#########################
mutable struct QCongrInvInt
   a::Int64 
   b::Int64 
   n::Int64 
   x::Union{Missing,Array{Int64,1}}
end 

# Data type for Q7_3
#####################################
# Matrix Inverse: Co-factors
#####################################
mutable struct QMInvCof
   n :: Int64
   A :: Array{Int64,2}
   Ainv :: Union{Missing,Array{Int64,2}}
end

# Data type for Q8_1
###########################
# Frequency Analysis
###########################
mutable struct QFreqAnalysis
    aStr :: String
    lowercase_count :: Union{Missing,Vector{Int}}
    uppercase_count :: Union{Missing,Vector{Int}}
end 

# Data type for Q8_2
###########################
# Vigenere Cipher
###########################
mutable struct QVigCipher
    English_f0 :: Vector{Float64}
    key :: Union{Missing,String} 
    p_text :: Union{Missing,String}
    c_text :: Union{Missing,String}
    f_c :: Union{Missing,Vector{Float64}}
    c_f :: Union{Missing,Float64}
    scf :: Union{Missing,Vector{Float64}}
    f_l :: Union{Missing,Vector{Vector{Float64}}}
    c_f_l :: Union{Missing,Vector{Float64}}
end

# Data type for Q8_3
#################################
# Block Cipher
#################################
mutable struct QBlockCipher
    A :: Union{Missing,Array{Int,2}}
    Ainv :: Union{Missing,Array{Int,2}}
    b :: Union{Missing,Vector{Int}}
    L :: Int
    p_text :: Union{Missing,String}
    c_text :: Union{Missing,String}
    p_text_known :: String
    c_text_known :: String
    f0 :: Vector{Float64}
    f_l :: Union{Missing,Vector{Vector{Float64}}}
    c_f_l :: Union{Missing,Vector{Float64}}
end


### Load Your Data

In [4]:
fdata_name = "CCT-CW1-06124635.jld2";

#-----------------------------------------------------------   Q7_1   -----------------------------------------------------------
    ans_Q7_1 = load(fdata_name, "ans_Q7_1")
    ans_Q7_1.c1 = mod(ans_Q7_1.b^ans_Q7_1.a, ans_Q7_1.n);
    ans_Q7_1.c2 = Int(mod(BigInt(ans_Q7_1.b)^ans_Q7_1.a, ans_Q7_1.n));
    ans_Q7_1.c3 = powermod(ans_Q7_1.b, ans_Q7_1.a, ans_Q7_1.n);
    ans_Q7_1
#The first method does not return the correct result because it first calculates b^a, which 
#causes an integer overflow, and then computes b^a mod n.
#The second method returns the correct result because b^a is stored using arbitrary-precision 
#memory (BigInt).
#The third method is the most efficient because it applies modular arithmetic properties, such 
#as (x*y) mod n=(x mod n)*(y mod n) mod n, to compute the result directly without generating large
#intermediate numbers


#-----------------------------------------------------------   Q7_2   -----------------------------------------------------------
#This function finds all solutions x in the range 0<=x<n such that a*x=b (mod n) 
#If solution exist, they are returned sorted in ascending order (x[1]<x[2]<...)
function modularDiv(a::Integer, b::Integer, n::Integer)
    #Input validation
    if (n <= 1 || a>=n)
        println("Invalid arguments")
        return Int[]
    end

    #Special case: a==0
    if (a == 0)
        if (b % n == 0) #0*x=0 (mod n) holds for all x in {1..n-1}
           return collect(1:n-1)
        else #0*x=b (mod n) with b!=0 has no solutions
            return Int[]
        end
    end

    #Solvability criterion: gcd(a,n) must divide b
    d = gcd(a,n)
    if (b % d != 0) #there are no solutions
        return Int[]
    else
        #Reduce the congruence (a/d)*x=b/d(mod n/d)
        a1 = div(a,d)
        b1 = div(b,d)
        n1 = div(n,d)

        #Compute the modular inverse of a1 modulo n1 and find the base solution x0
        y = invmod(a1, n1)
        b1 = mod(b1, n1)
        x0 = Int(mod(BigInt(y * b1), n1))

        res = Int[]
        start = 0
        if (x0 == 0) #if the base solution is equal to 0, you don't have to insert it in the solutions
            start = 1
        end

        #Generate all d distinct solutions 
        for i in start:(d-1)
            push!(res, x0 + i * n1)
        end
        
        return res
    end
        
end


    ans_Q7_2 = load(fdata_name, "ans_Q7_2")
    for i in 1:5
        ans_Q7_2[i].x = modularDiv(ans_Q7_2[i].a, ans_Q7_2[i].b, ans_Q7_2[i].n)
    end

#Verification for ANS_Q7_2
#    for i in 1:5
#        for j in 1:length(ans_Q7_2[i].x)
#            println(mod(ans_Q7_2[i].a * ans_Q7_2[i].x[j], ans_Q7_2[i].n) == ans_Q7_2[i].b)
#        end
#    end

#    println(modularDiv(4,0,12)) #it must discard x0=0 and return [3,6,9]


#-----------------------------------------------------------   Q7_3   -----------------------------------------------------------
#This function takes a square integer matrix A and returns its adjugate A*
function adjugate(A::AbstractMatrix{<:Integer})
    
    #Input validation: check if the matrix is square
    dim = size(A,1)
    if (dim != size(A,2))
        error("Input is not a square matrix")
    end

    #Initialize the square matrix of the same dimensions
    adj = zeros(BigInt, size(A))

    #Convert input matrix to BigInt to ensure exact cofactor determinants without Float64 precision loss
    A_Big = BigInt.(A)
    
    for i in 1 : dim
        for j in 1: dim
            #Compute exact cofactors using BigInt submatrices
            adj[i,j] = (-1)^(i+j) * round(BigInt, det(A_Big[1:end .!= j, 1:end .!= i]))
        end
    end
    
    return adj
end


#This function computes the inverse of a square integer matrix A modulo n
#A^(-1)=Adj(A)/det(A) (mod n)
function modInvMatrix(A::AbstractMatrix{<:Integer}, n::Integer)
    
    #Input validation: check if A is a square matrix
    dim = size(A,1)
    if (dim != size(A,2))
        error("Input A is not a square matrix")
    end

    #Input validation: check if n>1
    if (n <= 1)
        error("Input n <= 1")
    end
        
    #Compute exact det(A) (mod n) using BigInt to prevent Float64 precision loss on large matrices
    determinant = Int(mod(round(BigInt,det(BigInt.(A))),n))



    #check if determinant and n are coprime. If determinant == 0 they aren't due to the fact that gcd(0,n)==n for all n
    if (gcd(determinant, n) == 1)

        #modular inverse of the determinant
        detInv = invmod(determinant, n)

        #adjugate matrix 
        adjInv = adjugate(A)

        #computing the adjugate with BigInt elements prevents integer overflow during scalar-matrix multiplication
        return Int.(mod.(detInv * adjInv, n))  
    else
        println("The matrix")
        display(A)
        println("is not invertible modulo", n, "\n")
        return Matrix{Int64}(undef, 0, 0)
    end
    
end



    ans_Q7_3 = load(fdata_name, "ans_Q7_3")
    for i in 1:3
        ans_Q7_3[i].Ainv = modInvMatrix(ans_Q7_3[i].A, ans_Q7_3[i].n)
    end

# Verification for ANS_Q7_3
#    for i in 1:3
#        dim = size(ans_Q7_3[i].Ainv,1)
#        if (dim!=0)
#            println(mod.(ans_Q7_3[i].A * ans_Q7_3[i].Ainv, ans_Q7_3[i].n) == Matrix(I, dim, dim))
#        else 
#            println("Ainv = []; gcd(det(A),n) = ", gcd(round(BigInt, det(BigInt.(ans_Q7_3[i].A))), ans_Q7_3[i].n))
#        end
#    end

#-----------------------------------------------------------   Q8_1   -----------------------------------------------------------
const ALPHABET = length('a':'z') #number of letters in the alphabet

#This function counts the occurrences of lowercase letters ('a' to 'z') in the string s 
#and returns a vector of integers of length ALPHABET
function lowerOccur(s::String)
    array = zeros(Int,ALPHABET)

    for char in s
        if ('a' <= char <= 'z')
            index = Int(char - 'a' + 1)
            array[index] += 1
        end
        #If char is not a lowercase letter, it is simply ignored
    end

    return array
end

#This function counts the occurrences of uppercase letters ('A' to 'Z') in the string s 
#and returns a vector of integers of length ALPHABET
function upperOccur(s::String)
    array = zeros(Int,ALPHABET)

    for char in s
        if ('A' <= char <= 'Z')
            index = Int(char - 'A' + 1)
            array[index] += 1
        end
        #If char is not an uppercase letter, it is simply ignored
    end

    return array
end


    ans_Q8_1 = load(fdata_name, "ans_Q8_1")
    ans_Q8_1.lowercase_count = lowerOccur(ans_Q8_1.aStr)
    ans_Q8_1.uppercase_count = upperOccur(ans_Q8_1.aStr)

#-----------------------------------------------------------   Q8_2_1   -----------------------------------------------------------
    ans_Q8_2_1 = load(fdata_name, "ans_Q8_2_1")


#--------------------------   1   --------------------------

#This function shift a valid lowercase letter 'c' by a key character 'k', and returns the encrypted uppercase letter
#N.B: it doesn't control that both arguments are valid lowercase letters
function shiftCipher(c::Char, k::Char)
    c = c - 'a'
    k = k - 'a'

    x = mod(c+k, ALPHABET)
    return x + 'A'
end

#This function takes a valid uppercase letter 'c' and a lowercase key character 'k', and returns the decrypted lowercase letter
#N.B: it doesn't control that both arguments are valid lowercase letters
function shiftInvCipher(c::Char, k::Char)
    c = c - 'A'
    k = k - 'a'
    
    x = mod(c-k, ALPHABET)
    return x + 'a'
end

#This function extract only letters from the key and convert them in lowercase
function cleanKey(key::String)
    clean = ""
    for c in key 
        if isletter(c) 
            clean = clean * lowercase(c)
        end
    end
    
    return clean
end


#This function Vigenère transform the text's characters included in set with the provided key. To do so, it calls the function specified in 
#shift_fn
function vigenereTransform(text::String, key::String, set::StepRange{Char, Int64}, shift_fn::Function)
    key = cleanKey(key)

    if isempty(key)
        error("The key must contain at least one valid letter")
    end


    output = ""
    index = 1
    for char in text
        #process only characters contained in set
        if char in set
            output = output * shift_fn(char, key[index])

            #Advance key index cyclically only when a valid letter is encountered
            index = mod(index, length(key)) + 1;
        end
        #characters not included in set are simply omitted
    end
    return output
end

#This function encrypts the plaintext using the Vigenére cipher with the provided key. It returns the ciphertext in uppercase format
function vigenereEncrypt(plaintext::String, key::String)
    return vigenereTransform(plaintext, key, 'a':'z', shiftCipher)
end

#This function decrypts the ciphertext using the Vigenére cipher with the provided key. It returns the plaintext in lowercase format
function vigenereDecrypt(ciphertext::String, key::String)
    return vigenereTransform(ciphertext, key, 'A':'Z', shiftInvCipher)
end

    ans_Q8_2_1.c_text = vigenereEncrypt(ans_Q8_2_1.p_text, ans_Q8_2_1.key)
#--------------------------   2   --------------------------
#This function returns a vector whose elements represent uppercase letter frequencies
letterFrequencies(s::String) = upperOccur(s)/length(s)

    ans_Q8_2_1.f_c = letterFrequencies(ans_Q8_2_1.c_text)

    ans_Q8_2_1.c_f = dot(sort(ans_Q8_2_1.English_f0), sort(ans_Q8_2_1.f_c))
    max = dot(ans_Q8_2_1.English_f0, ans_Q8_2_1.English_f0)
    min = dot(ans_Q8_2_1.English_f0, fill(1/ALPHABET, ALPHABET))

#    println("Computed inner product:", ans_Q8_2_1.c_f)
#    println("Max inner product:", max)
#    println("Min inner product:", min, "\n")

#=
If a plaintext is encrypted only using a single monoalphabetic cipher and the ciphertext is long enough, the inner product between 
the standard English and the sorted cyphertext frequencies is around ||f_0||^2= 0.065601
This example shows that directly applying global frequency analysis to a Vigenère ciphertext is problematic: because multiple shift 
keys are mixed, the computed inner product drops significantly towards the theoretical minimum inner product, which is equal to the 
scalar product between the standard English frequencies and the uniform distribution
=#






#--------------------------   3   --------------------------

#This function computes the shifted coincidence frequencies for a given ciphertext and for shifts l in 1:L
function shiftedCoincidenceFreq(c_text::String, L_max::Int)
    dim = length(c_text)
    
    scf = zeros(L_max)
    
    for l in 1:L_max
        #count overlapping matching characters between the original and shifted cyphertext
        for i in 1 : (dim-l)
            if (c_text[i] == c_text[i+l])
                scf[l] += 1
            end
        end

        #normalize overlap countby the number f compared character pairs (dim - l)
        scf[l] = scf[l]/(dim-l)
        
    end
    
    return scf
end

    const L_max = 8
    ans_Q8_2_1.scf = shiftedCoincidenceFreq(ans_Q8_2_1.c_text, L_max)
#    println("Actual key length: ", length(ans_Q8_2_1.key))
#    println("Estimated key length: ", argmax(ans_Q8_2_1.scf), "\n")
    #the key length and the maximum argument of scf match, as we expected")






#--------------------------   4   --------------------------

#This function returns a vector containing the letter frequencies of all ciphertext substring 1<=l<=L and the inner product between 
#it (sorted) and the sorted letter frequency vector f0
function frequencyCorrelation(c_text::String, L::Int, f0::Vector{Float64})
    #vector of vectors of length L (non initialized arguments)
    f_l = Vector{Vector{Float64}}(undef, L)
    
    #vector of length L containing the inner products between letter frequency vectors f_0 and f_l
    c_f = Vector{Float64}(undef,L)
    
    dim = length(c_text)
    f0_sorted = sort(f0)
    
    for l in 1:L 
        f_l[l] = letterFrequencies(c_text[l:L:dim])
        c_f[l] = dot(f0_sorted, sort(f_l[l]))
    end
    return f_l, c_f
end


    ans_Q8_2_1.f_l, ans_Q8_2_1.c_f_l = frequencyCorrelation(ans_Q8_2_1.c_text, length(ans_Q8_2_1.key), ans_Q8_2_1.English_f0)

#    println("Correlation between the sorted frequency vectors of all ciphertext substring (ex 1): \n", ans_Q8_2_1.c_f_l,"\n")
#= 
The result shows that applying frequency analysis to Vigenère ciphertext sub-strings can help decryption because the inner 
product of each sorted sub-string frequency vector with the standard English frequencies is around ||f_0||^2= 0.065601, that 
is the expected value for a monoalphabetic cipher"
=#







#-----------------------------------------------------------   Q8_2_2   -----------------------------------------------------------
    ans_Q8_2_2 = load(fdata_name, "ans_Q8_2_2")

#--------------------------   1   --------------------------

    ans_Q8_2_2.f_c = letterFrequencies(ans_Q8_2_2.c_text)

    ans_Q8_2_2.c_f = dot(sort(ans_Q8_2_2.English_f0), sort(ans_Q8_2_2.f_c))
#Directly applying global frequency analysis to the text is ineffective for the reasons written above






#--------------------------   2   --------------------------
    ans_Q8_2_2.scf = shiftedCoincidenceFreq(ans_Q8_2_2.c_text, L_max)

    keyLength = argmax(ans_Q8_2_2.scf)






#--------------------------   3   --------------------------
    ans_Q8_2_2.f_l, ans_Q8_2_2.c_f_l = frequencyCorrelation(ans_Q8_2_2.c_text, keyLength, ans_Q8_2_2.English_f0)

#    println("Correlation between the sorted frequency vectors of all ciphertext substring (ex 2):\n", ans_Q8_2_2.c_f_l,"\n")

#The result shows that applying frequency analysis to Vigenère ciphertext sub-strings can help decryption for the reasons written above






#--------------------------   4   --------------------------

#This function takes the vector f_l, whose elements represent the letter frequencies of a cyphertext and the common letter frequency f_0
#Returns the estimated key (lowercase), which represents the argmax between the inner products of the cyclically shifted vector f_0 and the 
#letter frequency vector
function findKey(f_l::Vector{Vector{Float64}}, f_0::Vector{Float64})
    L = size(f_l)[1]
    key = ""
    
    for l in 1:L
        max = 0
        argmax = 0 #the inner product between the cyclically shifted English letter frequencies and the vector f_l[l] is >=0
        
        for i in 0:(ALPHABET-1)
            f_0_shifted = circshift(f_0, i)
            innProd = dot(f_0_shifted, f_l[l])
            if (innProd > max)
                max = innProd
                argmax = i
            end
 
        end

        key = key * (argmax + 'a')
        
    end
    return key
        
end

    ans_Q8_2_2.key = findKey(ans_Q8_2_2.f_l, ans_Q8_2_2.English_f0)
    ans_Q8_2_2.p_text = vigenereDecrypt(ans_Q8_2_2.c_text, ans_Q8_2_2.key)








#-----------------------------------------------------------   Q8_3   -----------------------------------------------------------
    ans_Q8_3 = load(fdata_name, "ans_Q8_3")

#--------------------------   1   --------------------------

    ans_Q8_3.f_l, ans_Q8_3.c_f_l = frequencyCorrelation(ans_Q8_3.c_text, ans_Q8_3.L, ans_Q8_3.f0)
    #println(ans_Q8_3.c_f_l)
#Since the correlation between the sorted frequency vectors is much lower than the theoretical maximum inner product, applying frequency
#analysis to break block ciphers is ineffective






#--------------------------   2   --------------------------


#This function takes a plaintext string, the corrispondent ciphertext and the length of each block L
#It returns the matrix A and the vector b used during the encrypt process
function hillAttack(p_text::String, c_text::String, L::Int)
    p_text_clear = [Int(c - 'a') for c in p_text if ('a'<=c<='z')]
    c_text_clear = [Int(c - 'A') for c in c_text if ('A'<=c<='Z')]
    
    n = length(p_text_clear) 
    
    #Input validation: p_text and c_text must have the same size and be divisible by L
    if(n != length(c_text_clear))
        error("The plaintext and the ciphertext must have the same length!")
    end
    if (mod(n,L)!=0)
        error("The plaintext lenghth is not a multiple of the block length")
    end

    num_blocks = div(n, L)
    
    P = reshape(p_text_clear, L, num_blocks)
    C = reshape(c_text_clear, L, num_blocks)

    
    deltaP = Matrix{Int}(undef, L, div(n,L)-1)
    deltaC = Matrix{Int}(undef, L, div(n,L)-1)
    
    for i in 1:size(P, 2)-1
        deltaP[:,i] = mod.(P[:,1] .- P[:,i+1], ALPHABET)
        deltaC[:,i] = mod.(C[:, 1] .- C[:,i+1], ALPHABET)
    end

    
    M = mod.(deltaP * transpose(deltaP), ALPHABET)
    invM = modInvMatrix(M, ALPHABET)
    if !isempty(invM)
        A = mod.(deltaC * transpose(deltaP) * invM, ALPHABET)
        b = mod.(C[:, 1] - A * P[:, 1], ALPHABET)
        return A, b
    end
end

#this function decrypt the c_text with the Hill cipher
function HillDecrypt(c_text::String, A_inv::Array{Int,2}, b::Vector{Int})
    #Input validation: A_inv must be a square matrix
    n = size(A_inv, 1)
    if (n != size(A_inv, 2))
        error("A_inv must be a square matrix")
    end

    #input validation: c_text must can be divided into blocks of size(A_inv,1) elements
    if (mod(length(c_text), n) != 0)
        error("c_text must can be divided into blocks of ",  n, " elements")
    end

    clean_c_text = [Int(c - 'A') for c in c_text if ('A'<=c<='Z')]
    A_inv_mod = mod.(A_inv, ALPHABET)
    
    p_text = ""
    L = div(length(c_text),size(A_inv,1)) #number of blocks
    for i in 0:(L-1)
        buffer= clean_c_text[i*n+1: (i+1)*n]
        buffer = mod.(A_inv_mod*(buffer - b), ALPHABET)

        for c in buffer
            p_text = p_text * Char(c + 'a')
        end
        
    end
    return p_text
end

    ans_Q8_3.A, ans_Q8_3.b = hillAttack(ans_Q8_3.p_text_known, ans_Q8_3.c_text_known, ans_Q8_3.L)
    ans_Q8_3.Ainv = modInvMatrix(ans_Q8_3.A, ALPHABET)
    ans_Q8_3.p_text = HillDecrypt(ans_Q8_3.c_text,ans_Q8_3.Ainv,ans_Q8_3.b)


The matrix


5×5 Matrix{Int64}:
 6  9  9  0   7
 5  6  2  8   8
 0  4  7  2   8
 0  5  7  7   9
 0  2  3  6  10

is not invertible modulo11



"againandyourfatherdidyoutellhimyesandheseemedtothinkwithmethatsomethinghadhappenedandthatishouldhearofhosmeragainashesaidwhatinterestcouldanyonehaveinbringingmetothedoorsofthechurchandthenleavingmenowifhehadborrowedmymoneyorifhehadmarriedmeandgotmymoneysettledonhimthere" ⋯ 3496 bytes ⋯ "partasthiswasithenglancedatherfaceandobservingthedintofapincenezateithersideofhernoseiventuredaremarkuponshortsightandtypewritingwhichseemedtosurpriseheritsurprisedmebutsurelyitwasobviousiwasthenmuchsurprisedandinterestedonglancingdowntoobservethatthoughthebootswhichshe"

## 2. Your Answers

### 7.1 Handling Large Integers

### 7.2 Congruence Inverse

### 7.3 Matrix Inverse

### 8.1 Frequency Analysis

### 8.2 Vigenere Cipher

### 8.3 Block Cipher

## 3. Save All Your Answers

Remember to save your answers into your data file!

In [5]:
   jldsave(fdata_name;
    ans_Q7_1, ans_Q7_2, ans_Q7_3,
     ans_Q8_1, ans_Q8_2_1, ans_Q8_2_2, ans_Q8_3
 );